In [1]:
# Operate from the parent directory
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import torch
from pose_encoder.advanced_pose_encoder import AdvancedPoseEncoder
from dataset.dataset import *
from utils.animation.skeleton import Skeleton
import utils.utils as utils
import matplotlib.pyplot as plt
import time

In [2]:
dataset = GPUDataset(
    consolidated_file="dataset/genea2023_dataset/toy/main-agent/consolidated.npz",
    seq_length=2000, # For testing
    seed_length=0,
    batch_size=1,
    epoch_length=1,  # Set to 1 for testing purposes
    return_audio_frame_index=True,  # Set to True to return the audio frame index
)

pose_encoder_test = AdvancedPoseEncoder.load_from_checkpoint("advanced_pose_encoder_ik_pca_64")

Initializing GPU-resident dataset on cuda
Loading data from dataset/genea2023_dataset/toy/main-agent/consolidated.npz directly to GPU...
Data loaded to GPU. Gesture shape: torch.Size([49628, 345]), Audio shape: torch.Size([49628, 37])
Found 28416 valid starting points for windows
Dataset initialization complete!

Unhandled bones that will use identity rotation:
  b_r_arm_twist: indices 57-62
  b_r_wrist_twist: indices 69-74
  b_l_arm_twist: indices 189-194
  b_l_wrist_twist: indices 201-206


In [3]:
# Import the viewer
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
animation_visualisation.init_visualization()
time.sleep(1)  # Wait for the visualization to initialize
animation_visualisation.send_character("ground_truth", (0.0,0.0,0.0), 0, "red")

Viewer URL: http://localhost:8001/utils/animation/visualisation/new/animation_viewer4.html?wsport=8702


In [12]:
dataset.reshuffle()

with torch.no_grad():
    # Get a sample from the dataset
    # gesture_sequence, seed_gesture, audio_features, main_agent_id_one_hot, full_audio_features, start_frame, file = next(iter(dataset))
    gesture_sequence, seed_gesture, audio_features, main_agent_id_one_hot, start_frames = next(iter(dataset))

    skeleton: Skeleton = dataset.skeleton

    # gesture_sequence = skeleton.denormalize_poses(gesture_sequence)

    print(f"Gesture sequence shape: {gesture_sequence.shape}")

    # Extract the first pose from the sequence
    first_pose = gesture_sequence[:,0].unsqueeze(0).float()

    print(f"First pose shape: {first_pose.shape}")

    # Encode the first pose
    encoded_pose = pose_encoder_test.encode(first_pose)

    # Decode the encoded pose back to the original pose space
    decoded_pose = pose_encoder_test.decode(encoded_pose)

    show = False

    if show:
        fig, axs = plt.subplots(1, 3, figsize=(18, 4))

        # First pose
        im0 = axs[0].imshow(first_pose.cpu().numpy().reshape(1, -1), aspect='auto', cmap='viridis', vmin=-2.5, vmax=2.5)
        axs[0].set_title("First Pose Tensor")
        axs[0].set_xlabel("Pose Dimension")
        axs[0].set_ylabel("Sample")
        # write the min and max values of the pose tensor
        axs[0].text(0.5, -0.1, f"Min: {first_pose.min().item():.2f}, Max: {first_pose.max().item():.2f}")
        plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

        # Encoded pose
        im1 = axs[1].imshow(encoded_pose.cpu().numpy().reshape(1, -1), aspect='auto', cmap='viridis', vmin=-2.5, vmax=2.5)
        axs[1].set_title("Encoded Pose Tensor")
        axs[1].set_xlabel("Encoded Dimension")
        axs[1].set_ylabel("Sample")
        # write the min and max values of the encoded pose tensor
        axs[1].text(0.5, -0.1, f"Min: {encoded_pose.min().item():.2f}, Max: {encoded_pose.max().item():.2f}")
        plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

        # Decoded pose
        im2 = axs[2].imshow(decoded_pose.cpu().numpy().reshape(1, -1), aspect='auto', cmap='viridis', vmin=-2.5, vmax=2.5)
        axs[2].set_title("Decoded Pose Tensor")
        axs[2].set_xlabel("Pose Dimension")

        axs[2].set_ylabel("Sample")
        # write the min and max values of the decoded pose tensor
        axs[2].text(0.5, -0.1, f"Min: {decoded_pose.min().item():.2f}, Max: {decoded_pose.max().item():.2f}")
        plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

        plt.tight_layout()
        plt.show()

    # print(.shape)
    animation_visualisation.send_pose(skeleton.denormalize_poses(first_pose).squeeze().cpu(), dataset.skeleton, "ground_truth")
    animation_visualisation.send_pose(skeleton.denormalize_poses(decoded_pose).squeeze().cpu(), dataset.skeleton)

    # The last 4x4 rows of the encoded pose are the IK target positions + swivel rotations (3 for position + 1 for swivel)
    # Extract the IK target positions from the encoded pose
    # ik_target_positions_and_swivel = encoded_pose[0, -4*4:].cpu().numpy()

    # remove the swivel rotation (the last 4th value in each 4D block)
    # ik_target_positions = ik_target_positions_and_swivel.reshape(-1, 4)[:, :3]  # Reshape to (4, 3) and remove swivel

    # Send IK target positions to the visualization
    # construct a tensor of shape (4, 3) from the ik_target_positions
    # ik_target_tensor = torch.tensor(ik_target_positions).reshape(4, 3)

    # print("IK Target Positions (without swivel):")
    # print(ik_target_tensor)

    # Extract encoded IK positions from the encoded pose
    # We know that these are the second last component encoded. That is, there are 6 * 4 values after the ik positions.
    # We also know that ik positions are encoded as 4D vectors (x, y, z, swivel).
    # print("Encoded pose shape:", encoded_pose.shape)
    # ik_reconstructed_pos = encoded_pose[0, 0, -6*4 - 4*4:-6*4].reshape(-1, 4)[:, :3]  # Reshape to (6, 3)

    # print("Ik_reconstructed Positions shape:", ik_reconstructed_pos.shape)

    # animation_visualisation.send_debug_positions(ik_reconstructed_pos)


    # first_pose_denormalized = skeleton.denormalize_poses(first_pose)
    # first_pose_world_positions = pose_encoder_test.skeleton.calculate_world_positions(first_pose_denormalized)
    # reshape to (1, num_joints, 3) for visualization
    # first_pose_world_positions = first_pose_world_positions.reshape(1, -1, 3)  # Reshape to (1, num_joints, 3)
    # print("First pose world positions shape:", first_pose_world_positions.shape)


    # animation_visualisation.send_debug_positions(first_pose_world_positions.squeeze().cpu())


Gesture sequence shape: torch.Size([1, 2000, 345])
First pose shape: torch.Size([1, 1, 345])

=== Pose Encoder Profiling Information ===
initialization                : 0.00 ms
preserved_components          : 1.00 ms
auto_encoded_components       : 1.00 ms
denormalization               : 0.00 ms
unhandled_indices             : 0.51 ms
identity_rotations            : 1.00 ms
world_positions               : 4.00 ms
ik_processing                 : 7.51 ms
world_preserve_after_ik       : 1.00 ms
final_normalization           : 1.00 ms


In [15]:
print(first_pose.squeeze().cpu().shape)
rest_pose = utils.get_rest_pose(58, device=utils.get_device())
print(rest_pose.cpu().shape)
animation_visualisation.send_pose(rest_pose.cpu(), skeleton=dataset.skeleton)

encoded_pose = pose_encoder_test.encode(rest_pose.unsqueeze(0).unsqueeze(0))

# The last 4x4 rows of the encoded pose are the IK target positions + swivel rotations (3 for position + 1 for swivel)
# Extract the IK target positions from the encoded pose
# ik_target_positions_and_swivel = encoded_pose[0, -4*4:].detach().cpu().numpy()

# remove the swivel rotation (the last 4th value in each 4D block)
# ik_target_positions = ik_target_positions_and_swivel.reshape(-1, 4)[:, :3]  # Reshape to (4, 3) and remove swivel

# Send IK target positions to the visualization
# construct a tensor of shape (4, 3) from the ik_target_positions
# ik_target_tensor = torch.tensor(ik_target_positions).reshape(4, 3)

# print("IK Target Positions (without swivel):")
# print(ik_target_tensor)

# animation_visualisation.send_debug_positions(ik_target_tensor)

def get_exact_bone_directions(skeleton, rest_pose, joint_name1, joint_name2):
    """Extract exact bone direction from rest pose."""
    # Calculate world positions in rest pose
    world_pos = skeleton.calculate_world_positions(rest_pose.unsqueeze(0))
    world_pos = world_pos.reshape(-1, len(skeleton.target_joints), 3)
    
    # Get joint indices
    idx1 = skeleton.target_joints.index(joint_name1)
    idx2 = skeleton.target_joints.index(joint_name2)
    
    # Get positions
    pos1 = world_pos[0, idx1].cpu()
    pos2 = world_pos[0, idx2].cpu()
    
    # Calculate exact normalized bone direction vector
    forward_dir = pos2 - pos1
    forward_dir = forward_dir / torch.norm(forward_dir)
    
    # Find perpendicular up direction (can use skeleton's up vector as reference)
    world_up = torch.tensor([0.0, 1.0, 0.0])
    right_dir = torch.linalg.cross(forward_dir, world_up)
    
    # Handle case where forward is aligned with world up
    if torch.norm(right_dir) < 1e-6:
        world_forward = torch.tensor([0.0, 0.0, 1.0])
        right_dir = torch.linalg.cross(world_forward, forward_dir)
    
    right_dir = right_dir / torch.norm(right_dir)
    up_dir = torch.linalg.cross(right_dir, forward_dir)
    up_dir = up_dir / torch.norm(up_dir)
    
    return {
        "bone": f"{joint_name1}->{joint_name2}",
        "forward_dir": forward_dir.tolist(),
        "up_dir": up_dir.tolist()
    }

print ("=== Exact Bone Directions ===")
print(get_exact_bone_directions(skeleton, rest_pose, 'b_l_upleg', 'b_l_leg'))
print(get_exact_bone_directions(skeleton, rest_pose, 'b_l_leg', 'b_l_foot'))
print ("----"*10)
print(get_exact_bone_directions(skeleton, rest_pose, 'b_r_upleg', 'b_r_leg'))
print(get_exact_bone_directions(skeleton, rest_pose, 'b_r_leg', 'b_r_foot'))
print ("===="*10)
print(get_exact_bone_directions(skeleton, rest_pose, 'b_l_arm', 'b_l_forearm'))
print(get_exact_bone_directions(skeleton, rest_pose, 'b_l_forearm', 'b_l_wrist_twist'))
print ("----"*10)
print(get_exact_bone_directions(skeleton, rest_pose, 'b_r_arm', 'b_r_forearm'))
print(get_exact_bone_directions(skeleton, rest_pose, 'b_r_forearm', 'b_r_wrist_twist'))

torch.Size([345])
torch.Size([345])
=== Exact Bone Directions ===
{'bone': 'b_l_upleg->b_l_leg', 'forward_dir': [0.0, -0.9999752044677734, 0.007038275245577097], 'up_dir': [0.0, 0.007038275711238384, 0.9999752640724182]}
{'bone': 'b_l_leg->b_l_foot', 'forward_dir': [0.0, -0.996783435344696, -0.0801420658826828], 'up_dir': [0.0, 0.0801420658826828, -0.996783435344696]}
----------------------------------------
{'bone': 'b_r_upleg->b_r_leg', 'forward_dir': [0.0, -0.9999904632568359, 0.004357356112450361], 'up_dir': [0.0, 0.0043573565781116486, 0.9999905228614807]}
{'bone': 'b_r_leg->b_r_foot', 'forward_dir': [0.0, -0.996783435344696, -0.0801420733332634], 'up_dir': [0.0, 0.0801420733332634, -0.996783435344696]}
{'bone': 'b_l_arm->b_l_forearm', 'forward_dir': [0.9993694424629211, -0.023177823051810265, -0.026897454634308815], 'up_dir': [0.02316943369805813, 0.9997313618659973, -0.0006235920009203255]}
{'bone': 'b_l_forearm->b_l_wrist_twist', 'forward_dir': [0.9993695020675659, -0.023177327